<a href="https://colab.research.google.com/github/anulax1114-design/PRAC1-st20347210-DA/blob/main/Streamlit_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4: Application Development — Streamlit


In [1]:
!pip install streamlit -q
print('Installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 63.1 MB/s eta 0:00:00
Installed


In [2]:
import os
os.makedirs('beijing_app/pages', exist_ok=True)
os.makedirs('beijing_app/.streamlit', exist_ok=True)

# Force light theme to fix dark background
with open('beijing_app/.streamlit/config.toml', 'w') as f:
    f.write("""[theme]
base = "light"
""")
print('Folders and theme config created')
!ls beijing_app

Folders and theme config created
pages


In [3]:
# ─── Download & Merge Beijing Air Quality Dataset ───────────────────────────
# Downloads all station CSV files from GitHub and merges them

import pandas as pd
import os
import urllib.request

# ── CONFIGURE YOUR GITHUB DETAILS HERE ──────────────────────────────────────
GITHUB_USER = 'anulax1114-design'   # e.g. 'anulax1114'
GITHUB_REPO = 'PRAC1-st20347210-DA'         # e.g. 'beijing-air-quality'
GITHUB_BRANCH = 'main'     # usually 'main' or 'master'
# ────────────────────────────────────────────────────────────────────────────

# Station CSV filenames — must match exactly what is in your GitHub repo
STATION_FILES = {
    'Dongsi'   : 'PRSA_Data_Dongsi_20130301-20170228.csv',
    'Guanyuan' : 'PRSA_Data_Guanyuan_20130301-20170228.csv',
    'Shunyi'   : 'PRSA_Data_Shunyi_20130301-20170228.csv',
    'Huairou'  : 'PRSA_Data_Huairou_20130301-20170228.csv',
}

BASE_URL = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/'

dfs = []
for station, filename in STATION_FILES.items():
    url = BASE_URL + filename
    print(f'Downloading {station} from {url} ...')
    try:
        df = pd.read_csv(url)
        df['station'] = station
        dfs.append(df)
        print(f'  ✅ {station}: {len(df):,} rows')
    except Exception as e:
        print(f'  ❌ Failed to download {station}: {e}')

if not dfs:
    raise RuntimeError('No station files downloaded. Check your GITHUB_USER, GITHUB_REPO, GITHUB_BRANCH and filenames above.')

merged = pd.concat(dfs, ignore_index=True)
merged.columns = [c.strip() for c in merged.columns]

# Fix TypeError: convert columns to numeric
numeric_cols = ['PM2.5','PM10','SO2','NO2','CO','O3','TEMP','PRES','DEWP','RAIN','WSPM']
for col in numeric_cols:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors='coerce')
merged['year'] = pd.to_numeric(merged['year'], errors='coerce')

out_path = '/content/beijing_merged_data.csv'
merged.to_csv(out_path, index=False)

print(f'\n✅ Merged dataset saved to {out_path}')
print(f'   Total rows : {len(merged):,}')
print(f'   Stations   : {merged["station"].unique().tolist()}')
print(f'   Columns    : {list(merged.columns)}')


  ✅ Dongsi: 35,064 rows
  ✅ Guanyuan: 35,064 rows
  ✅ Shunyi: 35,064 rows
  ✅ Huairou: 35,064 rows

✅ Merged dataset saved to /content/beijing_merged_data.csv
   Total rows : 140,256
   Stations   : ['Dongsi', 'Guanyuan', 'Shunyi', 'Huairou']
   Columns    : ['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']


In [4]:
%%writefile beijing_app/app.py
import streamlit as st
import pandas as pd
import os

st.set_page_config(page_title="Beijing Air Quality Explorer", page_icon="🌫️", layout="wide")

st.title("🌫️ Beijing Air Quality Explorer")
st.caption("CMP7005 · TASK 4 — Hourly monitoring data from 4 stations · March 2013 – February 2017")
st.divider()

col1, col2, col3 = st.columns(3)
with col1:
    st.info("**📋 DATASET**\n\nBrowse, filter, search and download the merged Beijing air quality dataset.")
with col2:
    st.info("**📊 VISUALISATION**\n\nKDE distributions, temporal trends, correlation heatmaps, seasonal comparisons.")
with col3:
    st.info("**🤖 MODEL OUTPUTS**\n\nRandom Forest diagnostics, feature importance and live PM2.5 prediction.")

st.divider()
st.info("👈 Use the sidebar to navigate between sections")

DATA_PATH = "beijing_merged_data.csv"

if not os.path.exists(DATA_PATH):
    st.subheader("📂 Upload Dataset")
    st.write("Upload your **beijing_merged_data.csv** file to unlock all pages.")
    uploaded = st.file_uploader("Upload CSV", type=["csv"])
    if uploaded is not None:
        df = pd.read_csv(uploaded)
        numeric_cols = ["PM2.5","PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM"]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df.to_csv(DATA_PATH, index=False)
        st.success("✅ Dataset uploaded! Please click any page in the sidebar to begin.")
else:
    df = pd.read_csv(DATA_PATH)
    numeric_cols = ["PM2.5","PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    st.subheader("Dataset Overview")
    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Total Records", f"{len(df):,}")
    c2.metric("Mean PM2.5", f"{df['PM2.5'].mean():.1f} µg/m³")
    c3.metric("Max PM2.5", f"{df['PM2.5'].max():.0f} µg/m³")
    c4.metric("Stations", str(df["station"].nunique()))
    c5.metric("Years", f"{int(df['year'].min())}–{int(df['year'].max())}")

    st.divider()
    if st.button("🗑️ Remove dataset and upload a new one"):
        os.remove(DATA_PATH)
        st.rerun()


Writing beijing_app/app.py


In [5]:
%%writefile beijing_app/pages/1_Dataset.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title='Dataset', page_icon='📋', layout='wide')

CSS = '<style>'\
    "@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;500;600&display=swap');"\
    'html,body,[class*=css]{font-family:DM Sans,sans-serif;}'\
    '.stApp{background:#0a0e1a;color:#e8eaf0;}'\
    "[data-testid='stSidebar']{background:linear-gradient(180deg,#0d1117,#0a0e1a);}"\
    "[data-testid='stSidebar'] *{color:#c9d1d9!important;}"\
    '.page-banner{background:linear-gradient(135deg,#1a2744,#0d1b3e);border:1px solid rgba(99,179,237,0.2);border-radius:16px;padding:24px 32px;margin-bottom:24px;}'\
    '.page-banner h1{font-family:Space Mono,monospace;font-size:1.4rem;color:#fff;margin:0 0 6px;}'\
    '.page-banner p{color:#8b9ab0;font-size:0.88rem;margin:0;}'\
    "div[data-testid='metric-container']{background:linear-gradient(135deg,#141e33,#0f1829);border:1px solid rgba(99,179,237,0.2);border-radius:12px;}"\
    "div[data-testid='metric-container'] label{color:#63b3ed!important;font-size:0.65rem!important;text-transform:uppercase!important;}"\
    "div[data-testid='metric-container'] [data-testid='stMetricValue']{color:#fff!important;font-size:1.6rem!important;font-weight:700!important;}"\
    '.insight-box{background:rgba(99,179,237,0.06);border-left:3px solid #63b3ed;border-radius:0 8px 8px 0;padding:12px 16px;margin:10px 0;font-size:0.84rem;color:#a0aec0;line-height:1.6;}'\
    '</style>'
st.html(CSS)

st.markdown("<div class='page-banner'><h1>📋 Dataset Explorer</h1><p>Browse, filter and summarise the merged Beijing air quality dataset</p></div>", unsafe_allow_html=True)

POLLUTANTS = ['PM2.5','PM10','SO2','NO2','CO','O3']
MET_VARS   = ['TEMP','PRES','DEWP','WSPM','RAIN']
STATIONS   = ['Dongsi','Guanyuan','Shunyi','Huairou']
SEASON_MAP = {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Autumn',10:'Autumn',11:'Autumn'}

DATA_PATH = 'beijing_merged_data.csv'
if not os.path.exists(DATA_PATH):
    st.error('Dataset not found.'); st.stop()

@st.cache_data
def load():
    df = pd.read_csv(DATA_PATH)
    df.columns = [c.strip() for c in df.columns]
    df['season'] = df['month'].map(SEASON_MAP)
    df['station_type'] = df['station'].apply(lambda s:'Urban' if s in ['Dongsi','Guanyuan'] else 'Suburban')
    return df

df = load()

with st.sidebar:
    st.markdown('### Filters')
    sel_stations = st.multiselect('Stations', STATIONS, default=STATIONS)
    sel_years = st.multiselect('Years', sorted(df['year'].unique()), default=sorted(df['year'].unique()))
    sel_seasons = st.multiselect('Seasons', ['Spring','Summer','Autumn','Winter'], default=['Spring','Summer','Autumn','Winter'])

dff = df[df['station'].isin(sel_stations)&df['year'].isin(sel_years)&df['season'].isin(sel_seasons)]
if dff.empty:
    st.warning('No data matches filters.'); st.stop()

c1,c2,c3,c4,c5 = st.columns(5)
c1.metric('Records', f'{len(dff):,}')
c2.metric('Mean PM2.5', f"{dff['PM2.5'].mean():.1f} µg/m³")
c3.metric('Max PM2.5', f"{dff['PM2.5'].max():.0f} µg/m³")
c4.metric('Stations', str(dff['station'].nunique()))
c5.metric('Date Range', f"{dff['year'].min()}–{dff['year'].max()}")

st.markdown('<br>', unsafe_allow_html=True)
tab1, tab2, tab3 = st.tabs(['🗂️  Raw Data', '📈  Statistics', '❓  Missing Values'])

with tab1:
    ca, cb = st.columns([3,1])
    search = ca.text_input('Search', '')
    n_rows = cb.select_slider('Rows', [50,100,250,500], value=100)
    show_cols = st.multiselect('Columns', dff.columns.tolist(),
        default=['year','month','day','hour','station','station_type','PM2.5','PM10','SO2','NO2','CO','O3','TEMP','WSPM'])
    disp = dff[show_cols]
    if search:
        disp = disp[disp.apply(lambda r:r.astype(str).str.contains(search,case=False).any(),axis=1)]
    st.dataframe(disp.head(n_rows),
        use_container_width=True, height=400)
    st.caption(f'Showing {min(n_rows,len(disp)):,} of {len(dff):,} rows')
    st.download_button('Download CSV', dff[show_cols].to_csv(index=False).encode(), 'beijing_filtered.csv', 'text/csv')

with tab2:
    grp = st.selectbox('Variable group', ['All pollutants','All meteorological','Custom'])
    if grp=='All pollutants': num_cols=POLLUTANTS
    elif grp=='All meteorological': num_cols=MET_VARS
    else: num_cols=st.multiselect('Select', POLLUTANTS+MET_VARS, default=['PM2.5','NO2','TEMP'])
    if num_cols:
        st.dataframe(dff[num_cols].describe().T.round(3), use_container_width=True)
        pv = st.selectbox('Station comparison', num_cols, key='pv')
        st.dataframe(dff.groupby('station')[pv].agg(['mean','median','std','min','max']).round(2), use_container_width=True)

with tab3:
    miss = dff[POLLUTANTS+MET_VARS].isna().sum().reset_index()
    miss.columns=['Feature','Missing']; miss['%']=(miss['Missing']/len(dff)*100).round(2)
    miss = miss.sort_values('%',ascending=False)
    cm1,cm2 = st.columns([2,1])
    with cm1:
        BG='#0d1117'
        fig,ax = plt.subplots(figsize=(8,4)); fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
        colors_m=['#f56565' if p>5 else '#ed8936' if p>1 else '#68d391' for p in miss['%']]
        bars = ax.barh(miss['Feature'], miss['%'], color=colors_m, edgecolor='none')
        for bar,val in zip(bars,miss['%']):
            ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8, color='#718096')
        ax.set_xlabel('% Missing'); ax.set_title('Missing Value Rate', color='#e2e8f0', pad=10)
        ax.spines[['top','right']].set_visible(False); ax.spines[['left','bottom']].set_color('#2d3748')
        ax.tick_params(colors='#718096'); ax.xaxis.label.set_color('#718096')
        st.pyplot(fig, use_container_width=True); plt.close()
    with cm2:
        st.dataframe(miss, use_container_width=True, height=350)
    st.markdown("<div class='insight-box'>Missing values are most prevalent in gas pollutants (SO2, NO2, CO, O3). Gaps were imputed using forward-fill then backward-fill within each station group.</div>", unsafe_allow_html=True)


Writing beijing_app/pages/1_Dataset.py


In [6]:
%%writefile beijing_app/pages/2_Visualisation.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title='Visualisation', page_icon='📊', layout='wide')

CSS = '<style>'\
    "@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;500;600&display=swap');"\
    'html,body,[class*=css]{font-family:DM Sans,sans-serif;}'\
    '.stApp{background:#0a0e1a;color:#e8eaf0;}'\
    "[data-testid='stSidebar']{background:linear-gradient(180deg,#0d1117,#0a0e1a);}"\
    "[data-testid='stSidebar'] *{color:#c9d1d9!important;}"\
    '.page-banner{background:linear-gradient(135deg,#1a2744,#0d1b3e);border:1px solid rgba(99,179,237,0.2);border-radius:16px;padding:24px 32px;margin-bottom:24px;}'\
    '.page-banner h1{font-family:Space Mono,monospace;font-size:1.4rem;color:#fff;margin:0 0 6px;}'\
    '.page-banner p{color:#8b9ab0;font-size:0.88rem;margin:0;}'\
    '.insight-box{background:rgba(99,179,237,0.06);border-left:3px solid #63b3ed;border-radius:0 8px 8px 0;padding:12px 16px;margin:10px 0;font-size:0.84rem;color:#a0aec0;line-height:1.6;}'\
    "</style>"
st.html(CSS)

st.markdown("<div class='page-banner'><h1>📊 Visualisation Suite</h1><p>Distributions, temporal trends, correlations, seasonal and station comparisons</p></div>", unsafe_allow_html=True)

POLLUTANTS = ['PM2.5','PM10','SO2','NO2','CO','O3']
MET_VARS   = ['TEMP','PRES','DEWP','WSPM','RAIN']
STATIONS   = ['Dongsi','Guanyuan','Shunyi','Huairou']
SCOLORS    = {'Dongsi':'#f56565','Guanyuan':'#ed8936','Shunyi':'#4299e1','Huairou':'#38b2ac'}
SEASON_MAP = {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Autumn',10:'Autumn',11:'Autumn'}
BG = '#0d1117'

DATA_PATH = 'beijing_merged_data.csv'
if not os.path.exists(DATA_PATH):
    st.error('Dataset not found.'); st.stop()

@st.cache_data
def load():
    df = pd.read_csv(DATA_PATH)
    df.columns = [c.strip() for c in df.columns]
    df['season'] = df['month'].map(SEASON_MAP)
    df['station_type'] = df['station'].apply(lambda s:'Urban' if s in ['Dongsi','Guanyuan'] else 'Suburban')
    return df

df = load()

def dark_ax(ax, fig):
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#2d3748')
    ax.tick_params(colors='#718096', labelsize=9)
    ax.xaxis.label.set_color('#718096'); ax.yaxis.label.set_color('#718096')
    if ax.get_title(): ax.title.set_color('#e2e8f0')

with st.sidebar:
    st.markdown('### Filters')
    sel_stations = st.multiselect('Stations', STATIONS, default=STATIONS)
    sel_years = st.multiselect('Years', sorted(df['year'].unique()), default=sorted(df['year'].unique()))
    sel_seasons = st.multiselect('Seasons', ['Spring','Summer','Autumn','Winter'], default=['Spring','Summer','Autumn','Winter'])

dff = df[df['station'].isin(sel_stations)&df['year'].isin(sel_years)&df['season'].isin(sel_seasons)]
if dff.empty:
    st.warning('No data matches filters.'); st.stop()

tab1,tab2,tab3,tab4 = st.tabs(['🌡️  Distributions','📅  Temporal Trends','🔗  Correlations','🌿  Seasonal & Station'])

with tab1:
    rc1,rc2 = st.columns(2)
    poll_x = rc1.selectbox('X variable', POLLUTANTS+MET_VARS, index=0)
    poll_y = rc2.selectbox('Y variable (scatter)', POLLUTANTS+MET_VARS, index=1)
    col1,col2 = st.columns(2)
    with col1:
        fig,ax = plt.subplots(figsize=(6,4))
        for stn in sel_stations:
            sub = dff[dff['station']==stn][poll_x].dropna()
            if len(sub)>10:
                sns.kdeplot(sub, ax=ax, label=stn, fill=True, alpha=0.2, linewidth=2, color=SCOLORS.get(stn,'#718096'))
        ax.set_xlabel(poll_x); ax.set_ylabel('Density')
        ax.legend(fontsize=8, facecolor=BG, labelcolor='white', framealpha=0.8)
        dark_ax(ax,fig); st.pyplot(fig,use_container_width=True); plt.close()
    with col2:
        fig2,ax2 = plt.subplots(figsize=(6,4))
        data_bp = [dff[dff['station']==s][poll_x].dropna().values for s in sel_stations]
        if any(len(d)>0 for d in data_bp):
            bp = ax2.boxplot(data_bp, labels=sel_stations, patch_artist=True,
                medianprops=dict(color='#ffd700',linewidth=2),
                whiskerprops=dict(color='#4a5568'), capprops=dict(color='#4a5568'),
                flierprops=dict(marker='o',markersize=2,alpha=0.3))
            for patch,stn in zip(bp['boxes'],sel_stations):
                patch.set_facecolor(SCOLORS.get(stn,'#718096')); patch.set_alpha(0.7)
        ax2.set_ylabel(poll_x); dark_ax(ax2,fig2); st.pyplot(fig2,use_container_width=True); plt.close()
    st.markdown('---')
    sn = st.slider('Sample size', 1000, 20000, 5000, 1000)
    samp = dff.sample(min(sn,len(dff)), random_state=42)
    fig3,ax3 = plt.subplots(figsize=(12,4))
    for stn in sel_stations:
        s = samp[samp['station']==stn][[poll_x,poll_y]].dropna()
        ax3.scatter(s[poll_x], s[poll_y], alpha=0.4, s=8, color=SCOLORS.get(stn,'#718096'), label=stn, rasterized=True)
    ax3.set_xlabel(poll_x); ax3.set_ylabel(poll_y)
    ax3.legend(fontsize=9, facecolor=BG, labelcolor='white', framealpha=0.8)
    dark_ax(ax3,fig3); st.pyplot(fig3,use_container_width=True); plt.close()

with tab2:
    t_var = st.selectbox('Variable', POLLUTANTS+MET_VARS, key='tvar')
    t_mode = st.radio('View', ['Hourly (diurnal)','Monthly trend','Annual trend'], horizontal=True)
    fig,ax = plt.subplots(figsize=(12,4.5))
    if t_mode=='Hourly (diurnal)':
        grp = dff.groupby(['hour','station'])[t_var].mean().reset_index()
        for stn in sel_stations:
            s = grp[grp['station']==stn].sort_values('hour')
            ax.plot(s['hour'], s[t_var], color=SCOLORS.get(stn,'#718096'), linewidth=2.5, marker='o', markersize=4, label=stn)
        ax.set_xlabel('Hour of Day'); ax.set_xticks(range(0,24,2))
    elif t_mode=='Monthly trend':
        grp = dff.groupby(['year','month','station'])[t_var].mean().reset_index()
        grp['date'] = pd.to_datetime(grp[['year','month']].assign(day=1))
        for stn in sel_stations:
            s = grp[grp['station']==stn].sort_values('date')
            ax.plot(s['date'], s[t_var], color=SCOLORS.get(stn,'#718096'), linewidth=1.8, label=stn)
    else:
        grp = dff.groupby(['year','station'])[t_var].mean().reset_index()
        for stn in sel_stations:
            s = grp[grp['station']==stn]
            ax.plot(s['year'], s[t_var], color=SCOLORS.get(stn,'#718096'), linewidth=2.5, marker='s', markersize=8, label=stn)
    if t_var=='PM2.5':
        ax.axhline(15, color='#f56565', linestyle='--', linewidth=1.2, alpha=0.7, label='WHO 15 µg/m³')
    ax.set_ylabel(t_var); ax.set_title(f'{t_mode} — {t_var}')
    ax.legend(fontsize=9, facecolor=BG, labelcolor='white', framealpha=0.8)
    dark_ax(ax,fig); st.pyplot(fig,use_container_width=True); plt.close()
    st.markdown('---')
    st.markdown('**Hour × Month PM2.5 Heatmap**')
    hm_stn = st.selectbox('Station', sel_stations, key='hmstn')
    hm_data = dff[dff['station']==hm_stn].groupby(['month','hour'])['PM2.5'].mean().unstack(fill_value=0)
    fig_h,ax_h = plt.subplots(figsize=(14,5))
    sns.heatmap(hm_data, ax=ax_h, cmap='YlOrRd', linewidths=0.3, linecolor=BG, cbar_kws={'shrink':0.7})
    ax_h.set_xlabel('Hour'); ax_h.set_ylabel('Month'); ax_h.set_title(f'Hour x Month PM2.5 — {hm_stn}', color='#e2e8f0')
    ax_h.set_facecolor(BG); fig_h.patch.set_facecolor(BG); ax_h.tick_params(colors='#718096')
    st.pyplot(fig_h,use_container_width=True); plt.close()
    st.markdown("<div class='insight-box'>Winter morning peak (months 11-2, hours 7-10) where cold temperatures suppress vertical mixing and trap pollutants near the surface.</div>", unsafe_allow_html=True)

with tab3:
    corr_vars = st.multiselect('Variables', POLLUTANTS+MET_VARS, default=POLLUTANTS+['TEMP','WSPM','DEWP'])
    if len(corr_vars)>=2:
        corr = dff[corr_vars].corr()
        mask = np.triu(np.ones_like(corr, dtype=bool))
        fig,ax = plt.subplots(figsize=(10,8))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.3, ax=ax, vmin=-1, vmax=1, annot_kws={'size':8}, cbar_kws={'shrink':0.7})
        ax.set_title('Pearson Correlation Matrix', color='#e2e8f0', pad=12)
        ax.set_facecolor(BG); fig.patch.set_facecolor(BG); ax.tick_params(colors='#718096', labelsize=8)
        st.pyplot(fig,use_container_width=True); plt.close()
        st.markdown("<div class='insight-box'>PM2.5-PM10 (r=0.87) and PM2.5-CO (r=0.74) confirm shared combustion sources. O3 negatively correlates with NO2 (r=-0.52). TEMP negatively correlates with PM2.5 (r=-0.31).</div>", unsafe_allow_html=True)

with tab4:
    season_order = ['Spring','Summer','Autumn','Winter']
    s_var = st.selectbox('Variable', POLLUTANTS+MET_VARS, key='svar')
    col1,col2 = st.columns(2)
    with col1:
        sd = dff.groupby(['season','station'])[s_var].mean().reset_index()
        fig,ax = plt.subplots(figsize=(7,4.5))
        x=np.arange(len(season_order)); w=0.2
        for i,stn in enumerate(sel_stations):
            s=sd[sd['station']==stn].set_index('season')
            vals=[s.loc[se,s_var] if se in s.index else 0 for se in season_order]
            ax.bar(x+i*w-w*len(sel_stations)/2, vals, w, label=stn, color=SCOLORS.get(stn,'#718096'), alpha=0.85, edgecolor='none')
        ax.set_xticks(x); ax.set_xticklabels(season_order); ax.set_ylabel(f'Mean {s_var}')
        ax.legend(fontsize=8, facecolor=BG, labelcolor='white', framealpha=0.8)
        dark_ax(ax,fig); st.pyplot(fig,use_container_width=True); plt.close()
    with col2:
        td = dff.groupby(['station_type','season'])[s_var].mean().unstack().reindex(columns=season_order)
        fig2,ax2 = plt.subplots(figsize=(7,4.5))
        x2=np.arange(len(season_order)); w2=0.35
        ax2.bar(x2-w2/2, td.loc['Urban'] if 'Urban' in td.index else [0]*4, w2, label='Urban', color='#f56565', alpha=0.85)
        ax2.bar(x2+w2/2, td.loc['Suburban'] if 'Suburban' in td.index else [0]*4, w2, label='Suburban', color='#4299e1', alpha=0.85)
        ax2.set_xticks(x2); ax2.set_xticklabels(season_order); ax2.set_ylabel(f'Mean {s_var}')
        ax2.legend(fontsize=9, facecolor=BG, labelcolor='white', framealpha=0.8)
        dark_ax(ax2,fig2); st.pyplot(fig2,use_container_width=True); plt.close()
    st.markdown("<div class='insight-box'>Winter PM2.5 peaks across all stations (urban ~95 vs suburban ~70 µg/m3). The urban-suburban gap is widest in winter and narrowest in summer.</div>", unsafe_allow_html=True)


Writing beijing_app/pages/2_Visualisation.py


In [7]:
%%writefile beijing_app/pages/3_Model_Outputs.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title='Model Outputs', page_icon='🤖', layout='wide')

CSS = '<style>'\
    "@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;500;600&display=swap');"\
    'html,body,[class*=css]{font-family:DM Sans,sans-serif;}'\
    '.stApp{background:#0a0e1a;color:#e8eaf0;}'\
    "[data-testid='stSidebar']{background:linear-gradient(180deg,#0d1117,#0a0e1a);}"\
    "[data-testid='stSidebar'] *{color:#c9d1d9!important;}"\
    '.page-banner{background:linear-gradient(135deg,#1a2744,#0d1b3e);border:1px solid rgba(99,179,237,0.2);border-radius:16px;padding:24px 32px;margin-bottom:24px;}'\
    '.page-banner h1{font-family:Space Mono,monospace;font-size:1.4rem;color:#fff;margin:0 0 6px;}'\
    '.page-banner p{color:#8b9ab0;font-size:0.88rem;margin:0;}'\
    "div[data-testid='metric-container']{background:linear-gradient(135deg,#141e33,#0f1829);border:1px solid rgba(99,179,237,0.2);border-radius:12px;}"\
    "div[data-testid='metric-container'] label{color:#63b3ed!important;font-size:0.65rem!important;text-transform:uppercase!important;}"\
    "div[data-testid='metric-container'] [data-testid='stMetricValue']{color:#fff!important;font-size:1.6rem!important;font-weight:700!important;}"\
    '.insight-box{background:rgba(99,179,237,0.06);border-left:3px solid #63b3ed;border-radius:0 8px 8px 0;padding:12px 16px;margin:10px 0;font-size:0.84rem;color:#a0aec0;line-height:1.6;}'\
    '.aqi-badge{display:inline-block;padding:5px 16px;border-radius:20px;font-weight:700;font-size:0.9rem;font-family:Space Mono,monospace;}'\
    "</style>"
st.html(CSS)

st.markdown("<div class='page-banner'><h1>🤖 Model Outputs</h1><p>Random Forest — diagnostics, feature importance, per-station performance and live PM2.5 prediction</p></div>", unsafe_allow_html=True)

MODEL_DIR = 'model_artefacts'
BG = '#0d1117'
AQI_BREAKS = [(0,12,'Good','#00e676','#000'),(12.1,35.4,'Moderate','#ffee58','#000'),
              (35.5,55.4,'Unhealthy (Sensitive)','#ffa726','#000'),
              (55.5,150.4,'Unhealthy','#ef5350','#fff'),
              (150.5,250.4,'Very Unhealthy','#ab47bc','#fff'),
              (250.5,9999,'Hazardous','#b71c1c','#fff')]

def aqi_info(v):
    for lo,hi,label,bg,fg in AQI_BREAKS:
        if lo<=v<=hi: return label,bg,fg
    return 'Hazardous','#b71c1c','#fff'

def dark_ax(ax, fig):
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#2d3748')
    ax.tick_params(colors='#718096', labelsize=9)
    ax.xaxis.label.set_color('#718096'); ax.yaxis.label.set_color('#718096')
    if ax.get_title(): ax.title.set_color('#e2e8f0')

@st.cache_resource
def load_model():
    try:
        gb    = joblib.load(os.path.join(MODEL_DIR,'gb_pm25_model.pkl'))
        le    = joblib.load(os.path.join(MODEL_DIR,'wind_label_encoder.pkl'))
        feats = joblib.load(os.path.join(MODEL_DIR,'feature_names.pkl'))
        preds = pd.read_csv(os.path.join(MODEL_DIR,'test_predictions.csv'))
        return gb,le,feats,preds
    except:
        return None,None,None,None

gb_model,le_wind,feat_names,test_preds = load_model()
if gb_model is None:
    st.warning('Model artefacts not found in model_artefacts/. Run Task 3 notebook first.')
    st.stop()

from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
actual=test_preds['actual']; predicted=test_preds['predicted']
rmse=np.sqrt(mean_squared_error(actual,predicted))
mae=mean_absolute_error(actual,predicted)
r2=r2_score(actual,predicted)

c1,c2,c3,c4 = st.columns(4)
c1.metric('Test RMSE', f'{rmse:.2f} µg/m³')
c2.metric('Test MAE',  f'{mae:.2f} µg/m³')
c3.metric('R² Score',  f'{r2:.4f}')
c4.metric('Test Samples', f'{len(actual):,}')

st.markdown('<br>', unsafe_allow_html=True)
tab1,tab2,tab3 = st.tabs(['📉  Diagnostics','🏆  Feature Importance','🔮  Live Prediction'])

with tab1:
    residuals = actual.values - predicted.values
    col1,col2 = st.columns(2)
    with col1:
        fig,ax = plt.subplots(figsize=(6,5))
        sc = ax.scatter(actual, predicted, alpha=0.12, s=6, rasterized=True, c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=100)
        plt.colorbar(sc, ax=ax, label='Abs Residual', shrink=0.8)
        mn,mx = actual.min(),actual.max()
        ax.plot([mn,mx],[mn,mx],'--',color='#f56565',linewidth=1.5,label='Perfect')
        ax.set_xlabel('Actual PM2.5 (µg/m³)'); ax.set_ylabel('Predicted PM2.5 (µg/m³)')
        ax.set_title('Actual vs Predicted')
        ax.legend(fontsize=8, facecolor=BG, labelcolor='white')
        dark_ax(ax,fig); st.pyplot(fig,use_container_width=True); plt.close()
    with col2:
        fig2,ax2 = plt.subplots(figsize=(6,5))
        ax2.hist(residuals, bins=80, color='#4299e1', edgecolor=BG, linewidth=0.3, alpha=0.85)
        ax2.axvline(0, color='#ffd700', linestyle='--', linewidth=1.5)
        ax2.axvline(np.mean(residuals), color='#f56565', linestyle='-', linewidth=1.2, label=f'Mean: {np.mean(residuals):.2f}')
        ax2.set_xlabel('Residual (µg/m³)'); ax2.set_ylabel('Count'); ax2.set_title('Residual Distribution')
        ax2.legend(fontsize=8, facecolor=BG, labelcolor='white')
        dark_ax(ax2,fig2); st.pyplot(fig2,use_container_width=True); plt.close()
    fig3,ax3 = plt.subplots(figsize=(12,3.5))
    ax3.scatter(predicted, residuals, alpha=0.1, s=5, color='#68d391', rasterized=True)
    ax3.axhline(0, color='#f56565', linestyle='--', linewidth=1.2)
    ax3.axhline(mae, color='#ffd700', linestyle=':', linewidth=1, label=f'+MAE ({mae:.1f})')
    ax3.axhline(-mae, color='#ffd700', linestyle=':', linewidth=1, label=f'-MAE ({mae:.1f})')
    ax3.set_xlabel('Predicted PM2.5 (µg/m³)'); ax3.set_ylabel('Residual'); ax3.set_title('Residuals vs Predicted')
    ax3.legend(fontsize=8, facecolor=BG, labelcolor='white')
    dark_ax(ax3,fig3); st.pyplot(fig3,use_container_width=True); plt.close()
    st.markdown("<div class='insight-box'>Residuals are approximately normally distributed and centred near zero, indicating an unbiased model.</div>", unsafe_allow_html=True)
    if 'station' in test_preds.columns:
        st.markdown('---')
        st.markdown('**Per-Station Performance**')
        stn_m = test_preds.groupby('station').apply(lambda g: pd.Series({
            'RMSE':np.sqrt(mean_squared_error(g['actual'],g['predicted'])),
            'MAE':mean_absolute_error(g['actual'],g['predicted']),
            'R2':r2_score(g['actual'],g['predicted']), 'n':len(g)
        })).reset_index()
        st.dataframe(stn_m.set_index('station').round(4), use_container_width=True)

with tab2:
    if hasattr(gb_model,'feature_importances_'):
        imp = pd.Series(gb_model.feature_importances_, index=feat_names).sort_values()
        top_n = st.slider('Top N features', 5, len(imp), min(15,len(imp)))
        top = imp.tail(top_n)
        fig,ax = plt.subplots(figsize=(9, max(4,top_n*0.45)))
        bar_colors = plt.cm.RdYlGn(top.values/top.values.max())
        top.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='none')
        for i,(val,name) in enumerate(zip(top.values,top.index)):
            ax.text(val+0.002, i, f'{val:.4f}', va='center', fontsize=8, color='#718096')
        ax.set_xlabel('Importance Score'); ax.set_title(f'Top {top_n} Feature Importances')
        dark_ax(ax,fig); st.pyplot(fig,use_container_width=True); plt.close()
        st.markdown("<div class='insight-box'>PM2.5_lag1 dominates at ~94.6%, confirming strong hourly autocorrelation. PM10 (4.4%) and CO (0.3%) reflect shared combustion sources.</div>", unsafe_allow_html=True)

with tab3:
    st.markdown('Enter current sensor and weather readings to predict PM2.5:')
    WIND_OPT = ['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW','W','WNW','NW','NNW','calm','CALM']
    with st.form('predict_form'):
        st.markdown('**Pollutant Readings**')
        rc1,rc2,rc3,rc4,rc5,rc6 = st.columns(6)
        pm10=rc1.number_input('PM10',    0.0,2000.0,80.0,step=1.0)
        so2 =rc2.number_input('SO2',     0.0,500.0, 15.0,step=1.0)
        no2 =rc3.number_input('NO2',     0.0,500.0, 50.0,step=1.0)
        co  =rc4.number_input('CO',      0.0,10000.0,600.0,step=10.0)
        o3  =rc5.number_input('O3',      0.0,500.0, 40.0,step=1.0)
        lag1=rc6.number_input('Prev PM2.5',0.0,1000.0,50.0,step=1.0)
        st.markdown('**Meteorological Conditions**')
        mc1,mc2,mc3,mc4 = st.columns(4)
        temp=mc1.number_input('Temp (C)',    -30.0,45.0,15.0,step=0.5)
        pres=mc2.number_input('Pressure',   980.0,1060.0,1010.0,step=0.5)
        dewp=mc3.number_input('Dew Point',  -40.0,30.0,5.0,step=0.5)
        wspm=mc4.number_input('Wind Speed', 0.0,20.0,2.0,step=0.1)
        ec1,ec2,ec3,ec4 = st.columns(4)
        rain    =ec1.number_input('Rainfall',0.0,100.0,0.0,step=0.1)
        wd      =ec2.selectbox('Wind Dir', WIND_OPT)
        stn_type=ec3.selectbox('Station Type', ['Urban','Suburban'])
        hour_in =ec4.slider('Hour',0,23,12)
        month_in=st.slider('Month',1,12,6)
        submitted=st.form_submit_button('PREDICT PM2.5', use_container_width=True)
    if submitted:
        is_urban=1 if stn_type=='Urban' else 0
        try: wd_enc=le_wind.transform([wd])[0]
        except: wd_enc=0
        feat_map={
            'PM10':pm10,'SO2':so2,'NO2':no2,'CO':co,'O3':o3,
            'TEMP':temp,'PRES':pres,'DEWP':dewp,'WSPM':wspm,'RAIN':rain,
            'wd_encoded':wd_enc,'is_urban':is_urban,
            'hour_sin':np.sin(2*np.pi*hour_in/24),
            'hour_cos':np.cos(2*np.pi*hour_in/24),
            'month_sin':np.sin(2*np.pi*month_in/12),
            'month_cos':np.cos(2*np.pi*month_in/12),
            'PM2.5_lag1':lag1
        }
        X_pred = np.array([[feat_map[f] for f in feat_names]])
        prediction = float(max(0.0, gb_model.predict(X_pred)[0]))
        label,bg_c,fg_c = aqi_info(prediction)
        st.markdown('---')
        res1,res2,res3 = st.columns([1,1,2])
        with res1:
            st.markdown(f"<div style='background:linear-gradient(135deg,#141e33,#0f1829);border:1px solid rgba(99,179,237,0.3);border-radius:14px;padding:24px;text-align:center;'>"
                f"<p style='font-size:0.6rem;letter-spacing:2px;color:#4a5568;margin:0 0 8px;'>PREDICTED PM2.5</p>"
                f"<p style='font-size:2.8rem;font-weight:700;color:#fff;margin:0;'>{prediction:.1f}</p>"
                f"<p style='color:#718096;font-size:0.8rem;margin:0;'>µg/m3</p></div>", unsafe_allow_html=True)
        with res2:
            st.markdown(f"<div style='background:linear-gradient(135deg,#141e33,#0f1829);border:1px solid rgba(99,179,237,0.3);border-radius:14px;padding:24px;text-align:center;'>"
                f"<p style='font-size:0.6rem;letter-spacing:2px;color:#4a5568;margin:0 0 12px;'>AQI CATEGORY</p>"
                f"<span class='aqi-badge' style='background:{bg_c};color:{fg_c};font-size:0.85rem;'>{label}</span></div>", unsafe_allow_html=True)
        with res3:
            fig_bar,ax_bar = plt.subplots(figsize=(7,1.3))
            thresholds=[0,12,35.4,55.4,150.4,250.4,350]
            aqi_colors=['#00e676','#ffee58','#ffa726','#ef5350','#ab47bc','#b71c1c']
            aqi_labels=['Good','Moderate','USG','Unhealthy','V.Unhealthy','Hazardous']
            for i,(lo,hi) in enumerate(zip(thresholds[:-1],thresholds[1:])):
                ax_bar.barh(0,hi-lo,left=lo,height=0.6,color=aqi_colors[i],edgecolor=BG,linewidth=0.5)
                ax_bar.text((lo+hi)/2,0,aqi_labels[i],ha='center',va='center',fontsize=6,color='black',fontweight='bold')
            ax_bar.axvline(min(prediction,340),color='white',linewidth=3,ymin=0.05,ymax=0.95)
            ax_bar.set_xlim(0,350); ax_bar.set_ylim(-0.5,0.5); ax_bar.axis('off')
            ax_bar.set_title(f'AQI Scale - {prediction:.1f} µg/m3',color='white',fontsize=9,pad=6)
            ax_bar.set_facecolor(BG); fig_bar.patch.set_facecolor(BG)
            st.pyplot(fig_bar,use_container_width=True); plt.close()
        advice={'Good':('OK','Air quality is satisfactory. Outdoor activities are safe.'),
                'Moderate':('Warning','Sensitive people should consider limiting outdoor exertion.'),
                'Unhealthy (Sensitive)':('Caution','Sensitive groups should reduce outdoor activity.'),
                'Unhealthy':('Alert','Everyone should limit prolonged outdoor exertion.'),
                'Very Unhealthy':('Health Alert','Avoid all outdoor physical activity.'),
                'Hazardous':('Emergency','Stay indoors. Avoid all outdoor activity.')}
        lvl,msg = advice.get(label,('Info','Monitor conditions.'))
        st.info(f'{lvl}: {msg}')


Writing beijing_app/pages/3_Model_Outputs.py


In [8]:
# Get IP for localtunnel password
!wget -q -O - ipv4.icanhazip.com

34.182.203.101


In [ ]:
# Run the app — open the URL printed below, use IP above as password
!streamlit run beijing_app/app.py &>/dev/null &
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://calm-zoos-repair.loca.lt


In [ ]:
# Download the full app as a zip
import shutil
shutil.make_archive('/content/beijing_app','zip','/content','beijing_app')
from google.colab import files
files.download('/content/beijing_app.zip')